# Handoff test: cross-session resume, verified for real

Verifies the mechanism `run_phase0_pilot.py` will use for the real Phase 0 campaign
survives an ACTUAL Kaggle session boundary (`/kaggle/working` wiped between sessions),
not just a same-process rerun. Uses `kaggle_session.copy_in`/`copy_out_and_version`
(the same Kaggle-Dataset round-trip the real pilot uses) and
`handoff_test.check_session2_precondition`, which makes session 2 **fail loudly**
if it doesn't find session 1's completed jobs after copying in -- it will not
silently start over.

**This is the SAME notebook run twice, as two different Dataset versions** -- not two
different notebooks. See README's "Handoff test" section for the full click-by-click
steps. Summary:

1. Edit `SESSION = 1` below, **Save Version -> Save & Run All**. Wait for it to finish.
2. Create a Kaggle Dataset from this run's `/kaggle/working` output (one-time, see
   README) OR let `copy_out_and_version` create it automatically (it calls
   `kaggle datasets create` if the dataset doesn't exist yet).
3. **Add Input -> your own Datasets -> the dataset from step 2**, attach it to this
   notebook.
4. Edit `SESSION = 2` below, **Save Version -> Save & Run All** again.
5. Read the last cell's printed verdict: should be `PASS: final set equals the job
   list exactly, no duplicates, nothing lost.`

## Settings to click

- **Accelerator: CPU only is fine** -- this test uses the mock LLM/env backend
  (zero-cost, verifies orchestration, not model behavior), no GPU needed.
- **Internet: ON** -- needed to clone the repo and for the `kaggle` CLI's Dataset
  push/pull.
- **Persistence: No persistence** is fine -- state is carried by the attached Dataset,
  not Kaggle's own session persistence option.

## Settings -- edit SESSION between the two runs (and DATASET_SLUG once)

In [ ]:
DATASET_SLUG = "your-kaggle-username/memory-pilot-handoff-test"  # EDIT THIS once
SESSION = 1  # EDIT to 2 for the second run, after attaching session 1's output Dataset


## 0. Setup -- clone and install (no PYTHONPATH anywhere)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!git clone https://github.com/ibo-agentic/memory-pilot.git
%cd memory-pilot
!git rev-parse HEAD


In [ ]:
# Editable-installs both packages so alfworld_pilot and memory_ope import in
# EVERY process (including run_jobs.py's own subprocess calls) with no PYTHONPATH
# needed anywhere -- verified locally (see README's Orchestration section).
!pip install -e . --quiet
!pip install -e alfworld_pilot --quiet
!pip install --upgrade kaggle --quiet


In [ ]:
import os

# Kaggle's newer single-token API auth (KAGGLE_API_TOKEN, format KGAT_...)
# replaces the old kaggle.json username/key pair. Read it from Kaggle
# Secrets -- never hardcode it, never print it.
try:
    from kaggle_secrets import UserSecretsClient
    _kaggle_api_token = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
except Exception as e:
    raise SystemExit(
        'FAIL: could not read the KAGGLE_API_TOKEN secret -- add it via this '
        'notebook\'s Add-ons -> Secrets menu (label exactly KAGGLE_API_TOKEN, '
        f'value starting KGAT_...) before running this cell. Error: {e}'
    )

os.environ['KAGGLE_API_TOKEN'] = _kaggle_api_token
del _kaggle_api_token  # don't leave the token sitting in a notebook-visible variable
print('KAGGLE_API_TOKEN loaded from Kaggle Secrets and set as an environment variable (value not printed).')


## 1. Import check -- stops the notebook here if anything failed to install

In [ ]:
try:
    import alfworld_pilot
    import memory_ope
    from alfworld_pilot import job_list, run_jobs, kaggle_session, handoff_test  # noqa: F401
    print('Import check PASSED:')
    print('  alfworld_pilot ->', alfworld_pilot.__file__)
    print('  memory_ope     ->', memory_ope.__file__)
except ImportError as e:
    raise SystemExit(f'Import check FAILED: {e!r} -- fix the install cell above before proceeding.')


## 2. Run this session

Calls `handoff_test.main()` directly (in-process, not via a `PYTHONPATH=...` shell
command) with the `SESSION` number set above.

In [ ]:
import sys
sys.argv = ['handoff_test', '--dataset-slug', DATASET_SLUG, '--session', str(SESSION), '--max-new', '3']
from alfworld_pilot import handoff_test
handoff_test.main()
